# Huấn luyện DETR-R50 · lượt so sánh chính · Signa v1.1

DETR-R50 gốc khởi tạo từ COCO, ảnh letterbox 640 × 640, tối đa 300 epoch,
dừng khi mAP@0.5:0.95 trên val không tăng sau 30 epoch, seed 0.
Learning rate giảm tuyến tính theo epoch với `LambdaLR`, `lrf = 0.01` và không dùng warmup.
Từ mốc 0 tới 300, LR đi từ 1e-4 đến 1e-6 cho nhóm chính và từ 1e-5 đến 1e-7 cho backbone.
[Giao thức huấn luyện](https://github.com/hana200804/DA/blob/5bcda664cac961a658011b9ed9094330d649865a/docs/04-thiet-ke-ai.md#3-giao-thức-huấn-luyện) ·
[Tăng cường dùng chung](https://github.com/hana200804/DA/blob/5bcda664cac961a658011b9ed9094330d649865a/ai/configs/augment.md).

1. Chọn **L4 GPU** trong Runtime → Change runtime type và đặt `signa-v1.1.zip` ở thư mục gốc My Drive.
2. Để kiểm tra luồng, đặt `SMOKE = True` ở ô đầu và chọn **Run all**. Smoke chạy một epoch với khoảng 2% của train và val, ghi vào `/content/signa-smoke/`.
3. Để chạy chính thức, đặt `SMOKE = False` và chọn **Run all**. Kết quả lưu ở `MyDrive/Signa/experiments/detr-r50-640-v1.1-main-s0/`.
4. Sau khi Colab ngắt, mở lại runtime L4 và chọn **Run all**. Notebook tiếp tục sau epoch đã ghi vào `last.pt` hoặc xuất val từ `best.pt` nếu đã huấn luyện xong. Mỗi lượt chỉ dùng một runtime cùng lúc.

Batch lấy mức cao nhất trong 16, 8, 4, 2, 1 vượt qua phép đo forward/backward AMP
ở 640 px, có dự trù trạng thái AdamW và 10% VRAM. Mức 16 là batch tổng tham chiếu
của DETR gốc trên 8 GPU. Notebook giữ batch đã lưu qua các phiên.
`meta.json` ghi thời gian giữ GPU, cập nhật theo epoch và khi kết thúc phiên.
Nếu Colab ngắt đột ngột, phần thời gian sau lần ghi gần nhất có thể thiếu.
Notebook bỏ qua batch có loss không hữu hạn và dừng nếu số batch bỏ qua vượt 1% của epoch.
Số lần bỏ qua được ghi theo epoch trong `results.csv` và theo phiên trong `meta.json`. Khi vượt ngưỡng, `meta.json` ghi epoch bị dừng.

Notebook xuất dự đoán và chấm trên val. Giao thức giữ tập test để đánh giá sau khi chốt cấu hình.

## 1. Cài đặt

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if "google.colab" not in sys.modules:
    raise RuntimeError("Notebook này chạy trên Colab với GPU L4.")

SMOKE = os.environ.get("SIGNA_SMOKE") == "1"  # Có thể đặt True để chạy thử trên Colab.
REPO_COMMIT = "5bcda664cac961a658011b9ed9094330d649865a"
REPO = Path("/content/signa-source")
TRANSFORMERS_VERSION = "4.57.6"
TIMM_VERSION = "1.0.24"
SCIPY_VERSION = "1.17.1"
PYCOCOTOOLS_VERSION = "2.0.11"


def sh(*args):
    subprocess.run([str(arg) for arg in args], check=True)


sh(
    sys.executable, "-m", "pip", "install", "-q",
    f"transformers=={TRANSFORMERS_VERSION}", f"timm=={TIMM_VERSION}",
    "albumentations==2.0.8", f"pycocotools=={PYCOCOTOOLS_VERSION}", f"scipy=={SCIPY_VERSION}",
)
if not (REPO / ".git").is_dir():
    sh("git", "clone", "-q", "https://github.com/hana200804/DA.git", REPO)
sh("git", "-C", REPO, "fetch", "-q", "origin")
sh("git", "-C", REPO, "checkout", "-q", "--detach", REPO_COMMIT)
sys.path.insert(0, str(REPO / "ai/src"))
os.environ["PYTHONPATH"] = str(REPO / "ai/src")

## 2. Kiểm tra dữ liệu

In [ ]:
import json
from tempfile import TemporaryDirectory
from zipfile import ZipFile

import cv2
import torch
import transformers
import yaml
from google.colab import drive

from signa_ai.dataset import relative_path
from signa_ai.detr import DetrDataset
from signa_ai.labels import load_label_map

assert transformers.__version__ == TRANSFORMERS_VERSION, "Khởi động lại runtime rồi chạy lại."
assert torch.cuda.is_available(), "Chọn GPU L4 trong phần cài đặt runtime."
GPU = torch.cuda.get_device_name()
assert "L4" in GPU, f"Run được cấu hình cho L4, runtime hiện tại: {GPU}"
DEVICE = torch.device("cuda:0")
print("GPU", GPU)
cv2.setNumThreads(0)

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive")
ARCHIVE = DRIVE / "signa-v1.1.zip"
DATASET = Path("/content/v1.1")
RUN_ROOT = Path("/content/signa-smoke") if SMOKE else DRIVE / "Signa/experiments"
RUN_DIR = RUN_ROOT / "detr-r50-640-v1.1-main-s0"

if not DATASET.is_dir():
    assert ARCHIVE.is_file(), f"Thiếu {ARCHIVE}"
    with TemporaryDirectory(dir=DATASET.parent) as staging, ZipFile(ARCHIVE) as archive:
        for member in archive.namelist():
            assert relative_path(member).parts[0] == DATASET.name, f"ZIP sai thư mục: {member}"
        archive.extractall(staging)
        (Path(staging) / DATASET.name).rename(DATASET)

LABELS = REPO / "ai/data/labels.yaml"
MANIFEST = DATASET / "manifest.csv"
label_map = load_label_map(LABELS)
NAMES = label_map.names
assert label_map.version == 1 and len(label_map) == 46
assert MANIFEST.read_bytes() == (REPO / "ai/data/manifests/v1.1.csv").read_bytes()
assert yaml.safe_load((DATASET / "data.yaml").read_text(encoding="utf-8"))["names"] == NAMES

train_set = DetrDataset(DATASET, "train", label_map)
val_set = DetrDataset(DATASET, "val", label_map)
for dataset, counts in ((train_set, (6679, 13283)), (val_set, (1599, 2685))):
    assert (len(dataset), len(dataset.truth["annotations"])) == counts
    for path, _ in dataset.images.values():
        assert (DATASET / path).is_file(), f"Thiếu ảnh {path}"
    print(f"{counts[0]} ảnh, {counts[1]} box")

## 3. Cấu hình và vòng lặp huấn luyện

In [ ]:
import math
import random

import numpy as np
from torch.utils.data import DataLoader, Subset, SubsetRandomSampler, get_worker_info
from tqdm.auto import tqdm
from transformers import DetrConfig, DetrForObjectDetection

from signa_ai.detection import EVAL_CONFIDENCE, to_coco_predictions
from signa_ai.detr import IMAGE_SIZE, detr_detections
from signa_ai.evaluate import evaluate

MODEL = "facebook/detr-resnet-50"
MODEL_REVISION = "1d5f47bd3bdd2c4bbfa585418ffe6da5028b4c0b"
SEED = 0
EPOCHS = 1 if SMOKE else 300
PATIENCE = 30
WORKERS = 2
LR = 1e-4
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-4
LRF = 0.01
CLIP = 0.1
MAX_NONFINITE_LOSS_FRACTION = 0.01
BATCH_CANDIDATES = (16, 8, 4, 2, 1)
BASE_CONFIG = {
    "repo_commit": REPO_COMMIT,
    "dataset_version": "v1.1",
    "label_map_version": label_map.version,
    "model": MODEL,
    "model_revision": MODEL_REVISION,
    "num_labels": len(NAMES),
    "image_size": IMAGE_SIZE,
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "seed": SEED,
    "workers": WORKERS,
    "optimizer": "AdamW",
    "lr": LR,
    "backbone_lr": BACKBONE_LR,
    "weight_decay": WEIGHT_DECAY,
    "lr_scheduler": "linear",
    "lrf": LRF,
    "clip_grad": CLIP,
    "amp": "float16",
    "max_nonfinite_loss_fraction": MAX_NONFINITE_LOSS_FRACTION,
    "auxiliary_loss": True,
    "augmentation": "signa_ai.augment.detr_transforms",
    "val_conf": EVAL_CONFIDENCE,
    "transformers": TRANSFORMERS_VERSION,
    "timm": TIMM_VERSION,
    "scipy": SCIPY_VERSION,
    "pycocotools": PYCOCOTOOLS_VERSION,
    "smoke": SMOKE,
}


def checkpoint_rng():
    algorithm, keys, *rest = np.random.get_state()
    return {
        "python": random.getstate(),
        "numpy": (algorithm, keys.tolist(), *rest),
        "cpu": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state(),
    }


def restore_rng(rng):
    random.setstate(rng["python"])
    np.random.set_state(rng["numpy"])
    torch.set_rng_state(rng["cpu"])
    torch.cuda.set_rng_state(rng["cuda"])


def seed_worker(_worker_id):
    # Albumentations giữ RNG riêng, nên seed của PyTorch phải được truyền cho từng worker.
    dataset = get_worker_info().dataset
    dataset.transform.set_random_seed(torch.initial_seed())


def collate(samples):
    pixels, masks, targets, image_ids, geometry = zip(*samples, strict=True)
    return (
        torch.from_numpy(np.stack(pixels)),
        torch.from_numpy(np.stack(masks)),
        [{key: torch.from_numpy(tensor) for key, tensor in target.items()} for target in targets],
        image_ids,
        geometry,
    )


def smoke_indices(dataset):
    return sorted(random.Random(SEED).sample(range(len(dataset)), math.ceil(0.02 * len(dataset))))


def make_loaders(batch):
    train_sampler = SubsetRandomSampler(smoke_indices(train_set)) if SMOKE else None
    train_loader = DataLoader(
        train_set, batch_size=batch, sampler=train_sampler, shuffle=train_sampler is None,
        num_workers=WORKERS, collate_fn=collate, worker_init_fn=seed_worker, pin_memory=True,
    )
    val_data = Subset(val_set, smoke_indices(val_set)) if SMOKE else val_set
    val_loader = DataLoader(
        val_data, batch_size=batch, num_workers=WORKERS, collate_fn=collate, pin_memory=True,
    )
    truth = val_set.truth
    if SMOKE:
        ids = {val_set.image_ids[index] for index in val_data.indices}
        truth = {
            **truth,
            "images": [row for row in truth["images"] if row["id"] in ids],
            "annotations": [row for row in truth["annotations"] if row["image_id"] in ids],
        }
    return train_loader, val_loader, truth


def build_model(pretrained):
    config = DetrConfig.from_pretrained(
        MODEL, revision=MODEL_REVISION, num_labels=46, id2label=NAMES,
        label2id={code: class_id for class_id, code in NAMES.items()},
        use_pretrained_backbone=False, auxiliary_loss=True,
    )
    assert config.num_labels == len(NAMES)
    model = (
        DetrForObjectDetection.from_pretrained(
            MODEL, revision=MODEL_REVISION, config=config,
            ignore_mismatched_sizes=True, use_safetensors=True,
        )
        if pretrained else DetrForObjectDetection(config)
    )
    return model.to(DEVICE)


def probe_memory(model, batch, box_count):
    pixels = torch.zeros((batch, 3, IMAGE_SIZE, IMAGE_SIZE), device=DEVICE)
    mask = torch.ones((batch, IMAGE_SIZE, IMAGE_SIZE), dtype=torch.bool, device=DEVICE)
    labels = {
        "class_labels": torch.zeros(box_count, dtype=torch.long, device=DEVICE),
        "boxes": torch.tensor([[0.5, 0.5, 0.25, 0.25]], device=DEVICE).repeat(box_count, 1),
    }
    with torch.autocast("cuda", dtype=torch.float16):
        loss = model(pixel_values=pixels, pixel_mask=mask, labels=[labels] * batch).loss
    loss.backward()
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated()


def choose_batch(model):
    model.train()
    rng = checkpoint_rng()
    box_count = max(map(len, train_set.annotations.values()))
    # AdamW giữ hai moment FP32 trên mỗi tham số học được.
    adam_bytes = sum(
        2 * p.numel() * p.element_size() for p in model.parameters() if p.requires_grad
    )
    budget = 0.9 * (torch.cuda.mem_get_info()[0] + torch.cuda.memory_reserved())
    for batch in BATCH_CANDIDATES:
        torch.cuda.reset_peak_memory_stats()
        try:
            fits = probe_memory(model, batch, box_count) + adam_bytes <= budget
        except torch.cuda.OutOfMemoryError:
            fits = False
        finally:
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            restore_rng(rng)
        if fits:
            print(f"Batch {batch} vừa ngân sách GPU {budget / 2**30:.1f} GiB.")
            return batch
    raise RuntimeError("GPU không đủ bộ nhớ cho DETR-R50 640 px với batch 1.")


def make_optimizer(model):
    backbone = set(model.model.backbone.parameters())
    return torch.optim.AdamW(
        [
            {"params": [p for p in model.parameters() if p.requires_grad and p not in backbone]},
            {"params": [p for p in model.parameters() if p.requires_grad and p in backbone],
             "lr": BACKBONE_LR},
        ],
        lr=LR, weight_decay=WEIGHT_DECAY, foreach=False,
    )


def make_scheduler(optimizer):
    return torch.optim.lr_scheduler.LambdaLR(
        optimizer, lr_lambda=lambda epoch: (1 - epoch / EPOCHS) * (1 - LRF) + LRF,
    )


def train_epoch(model, loader, optimizer, scaler, epoch):
    model.train()
    total = examples = skipped = 0
    skip_limit = int(len(loader) * MAX_NONFINITE_LOSS_FRACTION)
    for pixels, mask, targets, _, _ in tqdm(loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        pixels = pixels.to(DEVICE, non_blocking=True)
        mask = mask.to(DEVICE, non_blocking=True)
        targets = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in targets]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = model(pixel_values=pixels, pixel_mask=mask, labels=targets).loss
        value = loss.detach().item()
        if not math.isfinite(value):
            skipped += 1
            if skipped > skip_limit:
                return None, skipped
            continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        scaler.step(optimizer)
        scaler.update()
        total += value * len(pixels)
        examples += len(pixels)
    optimizer.zero_grad(set_to_none=True)
    return total / examples, skipped


@torch.inference_mode()
def predict_val(model, loader):
    model.eval()
    predictions = []
    for pixels, mask, _, image_ids, geometry in tqdm(loader, desc="Suy luận val"):
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = model(
                pixel_values=pixels.to(DEVICE, non_blocking=True),
                pixel_mask=mask.to(DEVICE, non_blocking=True),
            )
        batch_logits = outputs.logits.float().cpu().numpy()
        batch_boxes = outputs.pred_boxes.float().cpu().numpy()
        for image_id, shape, logits, boxes in zip(
            image_ids, geometry, batch_logits, batch_boxes, strict=True,
        ):
            detections = detr_detections(logits, boxes, shape, NAMES)
            predictions.extend(to_coco_predictions(image_id, detections, label_map))
    return predictions

## 4. Checkpoint, phục hồi và xuất kết quả val

In [ ]:
import csv
import hashlib
import time
from datetime import UTC, datetime
from importlib.metadata import distributions
from io import StringIO


def atomic_write(path, write):
    staged = path.with_name(f".{path.name}.tmp")
    with staged.open("wb") as stream:
        write(stream)
        stream.flush()
        os.fsync(stream.fileno())
    staged.replace(path)


def write_text(path, content):
    atomic_write(path, lambda stream: stream.write(content.encode("utf-8")))


def write_json(path, data):
    write_text(path, json.dumps(data, ensure_ascii=False, allow_nan=False, indent=2) + "\n")


def save_checkpoint(path, state):
    atomic_write(path, lambda stream: torch.save(state, stream))


def run_status(run_dir):
    if (run_dir / "done.json").is_file():
        return "done", None
    last = run_dir / "last.pt"
    if not last.is_file():
        return "new", None
    state = torch.load(last, map_location="cpu", weights_only=True)
    trained = state["epoch"] >= EPOCHS or state["patience_count"] >= PATIENCE
    return ("trained" if trained else "resume"), state


def sync_epoch_artifacts(run_dir, state):
    # last.pt là nguồn phục hồi nếu Colab ngắt giữa các lần ghi.
    best = state["best"]
    best_path = run_dir / "best.pt"
    if best["epoch"] == state["epoch"]:
        save_checkpoint(best_path, {"model": state["model"], "epoch": best["epoch"]})
    if not best_path.is_file():
        raise FileNotFoundError(f"Thiếu checkpoint đã chọn: {best_path}")
    write_json(run_dir / "best-selection.json", best)
    csv_file = StringIO(newline="")
    writer = csv.DictWriter(csv_file, fieldnames=list(state["history"][0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(state["history"])
    write_text(run_dir / "results.csv", csv_file.getvalue())


def start_session(run_dir):
    path = run_dir / "meta.json"
    meta = json.loads(path.read_text(encoding="utf-8")) if path.is_file() else {
        "repo_commit": REPO_COMMIT,
        "dataset_version": "v1.1",
        "label_map_version": label_map.version,
        "augmentation": BASE_CONFIG["augmentation"],
        "sessions": [],
    }
    session = {
        "started_at": datetime.now(UTC).isoformat(),
        "gpu": GPU,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "gpu_hours": 0.0,
        "skipped_nonfinite_loss_steps": 0,
    }
    meta["sessions"].append(session)
    write_json(path, meta)
    return meta, session, time.monotonic()


def load_run_config(run_dir, model, state):
    path = run_dir / "config.yaml"
    if path.is_file():
        config = yaml.safe_load(path.read_text(encoding="utf-8"))
    elif state is not None:
        config = state["config"]
    else:
        config = {**BASE_CONFIG, "batch": choose_batch(model)}
    if config != {**BASE_CONFIG, "batch": config["batch"]}:
        raise ValueError(f"Cấu hình hiện tại khác lượt đã lưu trong {run_dir}.")
    if state is not None and state["config"] != config:
        raise ValueError("Cấu hình không khớp checkpoint last.pt.")
    if not path.is_file():
        write_text(path, yaml.safe_dump(config, sort_keys=False, allow_unicode=True))

    requirements = run_dir / "requirements.txt"
    if not requirements.is_file():
        packages = {
            f"{name}=={dist.version}"
            for dist in distributions()
            if (name := dist.metadata["Name"])
        }
        write_text(requirements, "\n".join(sorted(packages)) + "\n")
    return config


def train(run_dir, model, state, config, train_loader, val_loader, truth, on_epoch):
    optimizer = make_optimizer(model)
    scheduler = make_scheduler(optimizer)
    scaler = torch.amp.GradScaler("cuda")
    last_epoch, best, patience_count, history = 0, None, 0, []
    if state is not None:
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        last_epoch = state["epoch"]
        best, patience_count, history = state["best"], state["patience_count"], state["history"]
        restore_rng(state["rng"])

    for epoch in range(last_epoch + 1, EPOCHS + 1):
        lr, backbone_lr = scheduler.get_last_lr()
        loss, skipped = train_epoch(model, train_loader, optimizer, scaler, epoch)
        if loss is None:
            on_epoch(epoch, skipped, completed=False)
            raise RuntimeError(
                f"Epoch {epoch}: {skipped}/{len(train_loader)} batch có loss không hữu hạn, "
                f"vượt ngưỡng {MAX_NONFINITE_LOSS_FRACTION:.0%}."
            )
        metrics = evaluate(truth, predict_val(model, val_loader), label_map)
        score = metrics["map_50_95"]
        if score is None or not math.isfinite(score):
            raise RuntimeError("Không tính được mAP@0.5:0.95 trên val.")
        if best is None or score > best["fitness"]:
            best = {
                "epoch": epoch,
                "metric": "map_50_95",
                "fitness": score,
                "val": {"map_50": metrics["map_50"], "map_50_95": score},
            }
            patience_count = 0
        else:
            patience_count += 1
        history.append({
            "epoch": epoch, "train_loss": loss, "map_50": metrics["map_50"], "map_50_95": score,
            "lr": lr, "backbone_lr": backbone_lr, "patience_count": patience_count,
            "skipped_nonfinite_loss_steps": skipped,
        })
        scheduler.step()
        state = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "epoch": epoch,
            "best": best,
            "patience_count": patience_count,
            "rng": checkpoint_rng(),
            "history": history,
            "config": config,
        }
        save_checkpoint(run_dir / "last.pt", state)
        on_epoch(epoch, skipped)
        sync_epoch_artifacts(run_dir, state)
        print(f"Epoch {epoch}: loss={loss:.4f}, mAP@0.5:0.95={score:.5f}, "
              f"patience={patience_count}, bỏ qua {skipped} batch")
        if patience_count >= PATIENCE:
            break
    return best


def export_val(run_dir, model, loader, truth, best):
    best_path = run_dir / "best.pt"
    state = torch.load(best_path, map_location="cpu", weights_only=True)
    assert state["epoch"] == best["epoch"], "best.pt không khớp epoch đã chọn."
    model.load_state_dict(state["model"])
    del state

    output = run_dir / "evaluation"
    output.mkdir(exist_ok=True)
    predictions = output / "predictions-val.json"
    write_json(predictions, predict_val(model, loader))
    with best_path.open("rb") as stream:
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    write_json(output / "export-config.json", {
        "weights_sha256": digest, "conf": EVAL_CONFIDENCE, "imgsz": IMAGE_SIZE,
        "dataset_version": "v1.1", "label_map_version": label_map.version,
    })
    ground_truth = DATASET / "coco/val.json"
    if SMOKE:
        ground_truth = output / "ground-truth-val-smoke.json"
        write_json(ground_truth, truth)

    metrics = run_dir / "metrics.json"
    staged = run_dir / ".metrics.json.tmp"
    sh(
        sys.executable, "-m", "signa_ai.evaluate", "--gt", ground_truth,
        "--pred", predictions, "--manifest", MANIFEST, "--labels", LABELS, "--out", staged,
    )
    staged.replace(metrics)
    report = json.loads(metrics.read_text(encoding="utf-8"))
    print(f"Val mAP@0.5:0.95 = {report['map_50_95']:.5f} từ epoch {best['epoch']}.")
    return report["map_50_95"]

## 5. Chạy hoặc tiếp tục lượt thực nghiệm

In [ ]:
def run(run_dir):
    status, state = run_status(run_dir)
    print(f"{run_dir.name}: {status}")
    if status == "done":
        return

    run_dir.mkdir(parents=True, exist_ok=True)
    meta, session, start = start_session(run_dir)
    session["last_epoch"] = state["epoch"] if state else 0

    def log_epoch(epoch=None, skipped=0, *, completed=True):
        if epoch is not None:
            session["skipped_nonfinite_loss_steps"] += skipped
            session["last_epoch" if completed else "aborted_epoch"] = epoch
        session["gpu_hours"] = (time.monotonic() - start) / 3600
        write_json(run_dir / "meta.json", meta)

    try:
        random.seed(SEED)
        np.random.seed(SEED)
        torch.manual_seed(SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        model = build_model(pretrained=status == "new")
        config = load_run_config(run_dir, model, state)
        print(f"Batch {config['batch']} trên {GPU}")
        train_loader, val_loader, truth = make_loaders(config["batch"])
        if state is not None:
            sync_epoch_artifacts(run_dir, state)
        if status == "trained":
            best = state["best"]
        else:
            if state is not None:
                model.load_state_dict(state["model"])
            best = train(run_dir, model, state, config, train_loader, val_loader, truth, log_epoch)
        del state
        val_map = export_val(run_dir, model, val_loader, truth, best)
    finally:
        session["ended_at"] = datetime.now(UTC).isoformat()
        log_epoch()

    write_json(run_dir / "done.json", {
        "finished_at": datetime.now(UTC).isoformat(),
        "best_epoch": best["epoch"],
        "map_50_95": val_map,
    })


run(RUN_DIR)
print("Kết quả:", RUN_DIR)